In [ ]:
from pathlib import Path
import colorsys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ======================== User configuration ========================
ROOT = Path("..")
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")
OUT = ROOT / "outputs" / "figures" / "figS1"
FIG_NAME = "Supplementary_Figure_S1_variable_distributions"
FONT_FAMILY = "Arial"
FIGSIZE = (9.6, 12.4)
POINT_SIZE, POINT_ALPHA = 2.0, 0.30
MAX_POINTS_PER_GROUP = 600
VIOLIN_ALPHA = 0.55
RANDOM_SEED = 20261116
# ==========================================================

VARIABLES = [
    ("bld_coverage", "Building density", "Building density"),
    ("bld_mean_h_m", "Mean building height (m)", "Mean building height (m)"),
    ("bld_std_h_m", "Building height SD (m)", "Building height SD (m)"),
    ("lcz_1", "LCZ 1: compact high-rise", "LCZ 1 fraction"),
    ("lcz_2", "LCZ 2: compact mid-rise", "LCZ 2 fraction"),
    ("lcz_3", "LCZ 3: compact low-rise", "LCZ 3 fraction"),
    ("ndvi_mean", "NDVI", "NDVI"),
    ("water_coverage", "Water coverage", "Water coverage"),
    ("isa_mean_prob", "Impervious surface cover", "Impervious surface cover"),
]
GROUPINGS = [
    ("A", "region", [("South Asia", "South\nAsia", "#F2C55C"), ("Southeast Asia", "Southeast\nAsia", "#F4C9BD"),
                     ("Africa", "Africa", "#BFE0D3"), ("Latin America", "Latin\nAmerica", "#C9B8E6")]),
    ("B", "koppen_broad", [("Tropical (A)", "Tropical\n(A)", "#F2C55C"), ("Arid (B)", "Arid\n(B)", "#F4C9BD"),
                           ("Temperate/Highland (C)", "Temperate\n(C)", "#BFE0D3")]),
]


def point_color(hex_color, lightness_drop=0.20, saturation_gain=1.0):
    h, l, sat = colorsys.rgb_to_hls(*mpl.colors.to_rgb(hex_color))
    return colorsys.hls_to_rgb(h, max(0.0, l - lightness_drop), min(1.0, sat * saturation_gain))


# ---------------- Load and check ----------------
p = pd.read_csv(PANEL, low_memory=False)
assert len(p) == 5216 and not p.duplicated(["city", "cell_id", "year"]).any() and p["city"].nunique() == 35
keep = p["sample_dml_controls"].astype(str).str.lower().eq("true")
d = p[keep].copy()
assert len(d) == 4576 and d.groupby(["city", "cell_id"]).ngroups == 572
for _, col, levels in GROUPINGS:
    assert set(d[col]) == {lv[0] for lv in levels}, f"Group values differ from expected: {col}"
for var, _, _ in VARIABLES:
    assert var in d.columns, f"Panel is missing variable: {var}"
frac = ["bld_coverage", "lcz_1", "lcz_2", "lcz_3", "water_coverage", "isa_mean_prob"]
assert all(d[v].dropna().between(0, 1).all() for v in frac), "Fraction variables should lie between 0 and 1"

# ---------------- Plotting ----------------
mpl.rcParams.update(mpl.rcParamsDefault)
font_manager.findfont(font_manager.FontProperties(family=FONT_FAMILY), fallback_to_default=False)
plt.rcParams.update({
    "font.family": FONT_FAMILY, "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.linewidth": 0.7,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})
rng = np.random.default_rng(RANDOM_SEED)
fig = plt.figure(figsize=FIGSIZE)
outer = fig.add_gridspec(2, 1, left=0.075, right=0.985, bottom=0.035, top=0.955, hspace=0.16)
stats = []
for block, (block_letter, group_col, levels) in enumerate(GROUPINGS):
    inner = outer[block].subgridspec(3, 3, hspace=0.50, wspace=0.27)
    for k, (var, title, ylabel) in enumerate(VARIABLES):
        ax = fig.add_subplot(inner[k // 3, k % 3])
        for pos, (level, tick, color) in enumerate(levels):
            v = d.loc[d[group_col].eq(level), var].dropna().to_numpy(float)
            assert len(v) >= 10, f"{var} has too few valid observations in {level}"
            if np.ptp(v) > 0:
                parts = ax.violinplot(v, positions=[pos], widths=0.82, showextrema=False)
                for body in parts["bodies"]:
                    body.set_facecolor(color); body.set_edgecolor("#7A7A7A")
                    body.set_linewidth(0.6); body.set_alpha(VIOLIN_ALPHA); body.set_zorder(1)
            shown = v if (MAX_POINTS_PER_GROUP is None or len(v) <= MAX_POINTS_PER_GROUP) else rng.choice(v, MAX_POINTS_PER_GROUP, replace=False)
            ax.scatter(pos + rng.uniform(-0.15, 0.15, len(shown)), shown, s=POINT_SIZE, color=point_color(color), alpha=POINT_ALPHA,
                       linewidths=0, zorder=2, rasterized=True)
            ax.boxplot(v, positions=[pos], widths=0.20, whis=1.5, showfliers=False, patch_artist=True, zorder=3,
                       boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.8),
                       medianprops=dict(color="black", linewidth=1.5),
                       whiskerprops=dict(color="black", linewidth=0.8), capprops=dict(color="black", linewidth=0.8))
            q25, med, q75 = np.percentile(v, [25, 50, 75])
            stats.append(dict(panel=f"{block_letter}{'abcdefghi'[k]}", grouping=group_col, group=level, variable=var,
                              label=title, n=len(v), n_points_shown=len(shown), n_cells=int(d.loc[d[group_col].eq(level) & d[var].notna()].groupby(["city", "cell_id"]).ngroups),
                              mean=float(v.mean()), sd=float(v.std(ddof=1)), min=float(v.min()), q25=float(q25),
                              median=float(med), q75=float(q75), max=float(v.max())))
        ax.set_xticks(range(len(levels)))
        ax.set_xticklabels([lv[1] for lv in levels])
        ax.set_xlim(-0.6, len(levels) - 0.4)
        ax.set_ylabel(ylabel)
        ax.set_title(f"({'abcdefghi'[k]}) {title}", fontweight="bold", pad=4)
        ax.grid(axis="y", color="#ECECEC", linewidth=0.55, zorder=0)
        ax.set_axisbelow(True)
        ax.tick_params(axis="x", length=0)
    box = outer[block].get_position(fig)
    fig.text(0.008, box.y1 + 0.012, block_letter, fontsize=17, fontweight="bold", ha="left", va="bottom")

# ---------------- Output ----------------
OUT.mkdir(parents=True, exist_ok=True)
stats = pd.DataFrame(stats)
assert len(stats) == len(VARIABLES) * sum(len(g[2]) for g in GROUPINGS) == 63
stats.to_csv(OUT / "variable_distribution_statistics.csv", index=False, encoding="utf-8-sig")
for ext in ["pdf", "svg", "png"]:
    fig.savefig(OUT / f"{FIG_NAME}.{ext}", dpi=300, facecolor="white")
plt.show()

print("Output:", OUT)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

# ======================== User configuration ========================
ROOT = Path("..")
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")
GRID_DIR = Path("../data/grids")
BOUNDARY_SHP = Path("../data/smod_0514.shp")
OUT = ROOT / "outputs" / "figures" / "figS2_S3"
FONT_FAMILY = "Arial"
CMAP = "RdBu_r"
MISSING_COLOR = "#CFCFCF"
FIGSIZE_S2 = (8.4, 10.9)
FIGSIZE_S3 = (10.4, 9.4)
# ==========================================================

CITIES = [
    ("Bangkok_Thailand", "Bangkok", "Thailand", "Southeast Asia", "Tropical", "Bangkok"),
    ("Jakarta_Indonesia", "Jakarta", "Indonesia", "Southeast Asia", "Tropical", "Jakarta"),
    ("Delhi_India", "Delhi", "India", "South Asia", "Arid", "Delhi"),
    ("Dhaka_Bangladesh", "Dhaka", "Bangladesh", "South Asia", "Tropical", "Dhaka"),
    ("Cairo_Egypt", "Cairo", "Egypt", "Africa", "Arid", "Cairo"),
    ("Johannesburg_SouthAfrica", "Johannesburg", "South Africa", "Africa", "Temperate", "Johannesburg"),
    ("MexicoCity_Mexico", "Mexico City", "Mexico", "Latin America", "Temperate", "Mexico City"),
    ("BuenosAires_Argentina", "Buenos Aires", "Argentina", "Latin America", "Temperate", "Buenos Aires"),
]
TYPES = ["day", "night", "compound"]
TYPE_LABELS = {"day": "Daytime-only", "night": "Nighttime-only", "compound": "Compound"}
TYPE_COLORS = {"day": "#D9902B", "night": "#3D7AB8", "compound": "#A8322D"}
KOPPEN_SHORT = {"Tropical (A)": "Tropical", "Arid (B)": "Arid", "Temperate/Highland (C)": "Temperate"}

# ---------------- Load and check ----------------
p = pd.read_csv(PANEL, low_memory=False)
assert len(p) == 5216 and not p.duplicated(["city", "cell_id", "year"]).any()
p["complete"] = p["heatwave_annual_complete"].astype(str).str.lower().eq("true")
hw_cols = [f"HW_days_{t}" for t in TYPES]
ex_cols = [f"exposure_people_days_{t}" for t in TYPES]
assert p.loc[p["complete"], hw_cols + ex_cols].notna().all().all(), "Grid-cell-years with complete heatwave records should have no missing values"
assert p.groupby(["city", "cell_id"])["complete"].nunique().eq(1).all(), "The completeness flag should be consistent across years for the same grid cell"
p["hwd_total"] = p[hw_cols].sum(axis=1, min_count=3)
p["exposure_total"] = p[ex_cols].sum(axis=1, min_count=3)
value_cols = ["hwd_total", "exposure_total"] + hw_cols + ex_cols
cell = p.groupby(["city", "cell_id"]).agg(**{c: (c, "mean") for c in value_cols}, n_years=("year", "nunique"),
                                         complete=("complete", "all"), region=("region", "first"),
                                         koppen=("koppen_broad", "first")).reset_index()
assert cell["n_years"].eq(8).all()
cell.loc[~cell["complete"], value_cols] = np.nan
for c in ["exposure_total"] + ex_cols:
    cell[c] = cell[c] / 1e6
bnd_all = gpd.read_file(BOUNDARY_SHP)
assert bnd_all.crs is not None and bnd_all.crs.to_epsg() == 4326
layers = {}
for city, disp, country, region, climate, bname in CITIES:
    g = gpd.read_file(GRID_DIR / f"{city}_era5_grid.gpkg")
    assert g.crs.to_epsg() == 4326 and g["cell_id"].is_unique
    c = cell[cell["city"].eq(city)]
    assert len(c) > 0 and set(c["cell_id"]) <= set(g["cell_id"]), f"Grid-cell vector lacks grid cells present in the panel: {city}"
    assert c["region"].iloc[0] == region and KOPPEN_SHORT[c["koppen"].iloc[0]] == climate, f"Region or climate label does not match the panel: {city}"
    g = g.merge(c.drop(columns="city"), on="cell_id", how="inner", validate="one_to_one")
    assert len(g) == len(c) and g["city"].eq(city).all()
    b = bnd_all[bnd_all["name"].eq(bname)]
    assert len(b) == 1, f"No unique match in the boundary file: {bname}"
    layers[city] = (g, b)
    assert (g.loc[g["complete"], "hwd_total"] > 0).all(), f"{city} has grid cells with zero heatwave days; the legend needs to be changed accordingly"

# ---------------- Plotting functions ----------------
mpl.rcParams.update(mpl.rcParamsDefault)
font_manager.findfont(font_manager.FontProperties(family=FONT_FAMILY), fallback_to_default=False)
plt.rcParams.update({"font.family": FONT_FAMILY, "font.size": 8.5,
                     "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"})


def draw_map(fig, ax, g, b, column, label=None, title=None, title_color="black"):
    v = g[column].to_numpy(float)
    ok = np.isfinite(v)
    assert ok.any(), f"No values to draw: {column}"
    vmin, vmax = float(v[ok].min()), float(v[ok].max())
    if vmax <= vmin:
        vmax = vmin + 1e-9
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    g[ok].plot(ax=ax, column=column, cmap=CMAP, norm=norm, edgecolor="white", linewidth=0.25, zorder=1)
    if (~ok).any():
        g[~ok].plot(ax=ax, color=MISSING_COLOR, edgecolor="white", linewidth=0.25, zorder=1)
    b.boundary.plot(ax=ax, color="black", linewidth=0.55, zorder=2)
    x0, y0, x1, y1 = g.total_bounds
    k = np.cos(np.deg2rad((y0 + y1) / 2))
    pos = ax.get_position()
    w_in, h_in = pos.width * fig.get_figwidth(), pos.height * fig.get_figheight()
    scale = max((x1 - x0) * k / w_in, (y1 - y0) / h_in) * 1.06
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    ax.set_xlim(cx - scale * w_in / k / 2, cx + scale * w_in / k / 2)
    ax.set_ylim(cy - scale * h_in / 2, cy + scale * h_in / 2)
    ax.set_aspect("auto")
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=8.5, fontweight="bold", color=title_color, pad=1.5)
    cax = ax.inset_axes([0.14, -0.085, 0.72, 0.045])
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=CMAP), cax=cax, orientation="horizontal")
    cb.set_ticks([vmin, vmax])
    digits = 1 if vmax >= 2 else 2
    cb.set_ticklabels([f"{vmin:.{digits}f}", f"{vmax:.{digits}f}"])
    cb.ax.tick_params(labelsize=7.5, length=2, width=0.5, pad=1.5)
    cb.outline.set_linewidth(0.5)
    if label:
        cax.set_xlabel(label, fontsize=7.8, labelpad=2)
    return vmin, vmax


def make_figure(name, figsize, columns, labels, titles=None, title_colors=None, note=None):
    n = len(columns)
    fig = plt.figure(figsize=figsize)
    top = 0.955 if note else 0.985
    outer = fig.add_gridspec(4, 2, left=0.012, right=0.988, bottom=0.03, top=top, hspace=0.07, wspace=0.03)
    ranges = []
    for i, (city, disp, country, region, climate, _) in enumerate(CITIES):
        g, b = layers[city]
        spec = outer[i // 2, i % 2]
        box = spec.get_position(fig)
        head = 0.030 * 10.9 / figsize[1]
        fig.add_artist(Rectangle((box.x0, box.y0), box.width, box.height, fill=False, edgecolor="black", linewidth=0.7))
        fig.add_artist(Line2D([box.x0, box.x1], [box.y1 - head, box.y1 - head], color="black", linewidth=0.7))
        fig.text(box.x0 + 0.012, box.y1 - head / 2, f"{disp} ({country}, {region}, {climate})",
                 fontsize=10.5, ha="left", va="center")
        inner = spec.subgridspec(1, n, wspace=0.04)
        for j, column in enumerate(columns):
            ax = fig.add_subplot(inner[0, j])
            pos = ax.get_position()
            pad_top = head + (0.020 if titles else 0.006)
            pad_bottom = (0.062 if labels[j] else 0.036) * 10.9 / figsize[1]
            ax.set_position([pos.x0 + 0.004, box.y0 + pad_bottom, pos.width - 0.008, box.height - pad_top - pad_bottom])
            vmin, vmax = draw_map(fig, ax, g, b, column, label=labels[j],
                                  title=titles[j] if titles else None,
                                  title_color=title_colors[j] if title_colors else "black")
            ranges.append(dict(figure=name, city=disp, variable=column, colour_min=vmin, colour_max=vmax,
                               n_cells=len(g), n_cells_without_temperature=int((~g["complete"]).sum())))
    if note:
        fig.text(0.012, 0.992, note, fontsize=9.5, ha="left", va="top")
    fig.legend(handles=[Patch(facecolor=MISSING_COLOR, edgecolor="white", label="No complete temperature record"),
                        Line2D([0], [0], color="black", linewidth=0.8, label="Urban study area")],
               loc="upper right", bbox_to_anchor=(0.99, 1.0 if note else 0.0), ncol=2, frameon=False, fontsize=8.5,
               handlelength=1.4, columnspacing=1.4) if note else None
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(OUT / f"{name}.{ext}", dpi=300, facecolor="white")
    plt.show()
    return ranges


# ---------------- Output ----------------
OUT.mkdir(parents=True, exist_ok=True)
all_ranges = []
all_ranges += make_figure(
    "Supplementary_Figure_S2_total_exposure_and_HWD", FIGSIZE_S2,
    columns=["exposure_total", "hwd_total"],
    labels=["Total exposure\n(million person-days yr$^{-1}$)", "Total HWD\n(days yr$^{-1}$)"],
    note="Mean annual values, 2016\u20132023")
all_ranges += make_figure(
    "Supplementary_Figure_S3_exposure_by_heatwave_type", FIGSIZE_S3,
    columns=ex_cols, labels=[None] * 3,
    titles=[TYPE_LABELS[t] for t in TYPES], title_colors=[TYPE_COLORS[t] for t in TYPES],
    note="Mean annual exposure by heatwave type (million person-days yr$^{-1}$), 2016\u20132023")

export = pd.concat([layers[c[0]][0].drop(columns="geometry").assign(city_label=c[1]) for c in CITIES], ignore_index=True)
keep = ["city", "city_label", "cell_id", "lat_center", "lon_center", "region", "koppen", "complete"] + value_cols
export[keep].rename(columns={"complete": "complete_temperature_record"}).to_csv(
    OUT / "city_map_plot_data.csv", index=False, encoding="utf-8-sig")
ranges = pd.DataFrame(all_ranges)
ranges.to_csv(OUT / "city_map_colour_ranges.csv", index=False, encoding="utf-8-sig")

print("Output:", OUT)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from scipy.stats import t

# ======================== User configuration ========================
ROOT = Path("..")
DML_RUN = Path("../outputs/dml")
OUT = ROOT / "outputs" / "figures" / "figS4_S10"
FIG_NAME = "Figure_DML_thresholds_full_sample"
INFERENCES = ["city_clustered", "conventional"]
PANEL_LETTERS = "ABCD"
FIGSIZE = (9.5, 6.9)
STAR_FONTSIZE = 8.5
XLIMS = {"exposure": (-0.80, 1.15), "HWD": (-0.16, 0.21)}
# ==========================================================

ORDER = ["local_p90", "local_p95", "city_p90", "city_p95", "fixed_tx35_tn25"]
LABELS = {
    "local_p90": "Cell-specific p90 (main)",
    "local_p95": "Cell-specific p95",
    "city_p90": "City-wide p90",
    "city_p95": "City-wide p95",
    "fixed_tx35_tn25": "Fixed thresholds (35/25 \u00b0C)",
}
COLORS = dict(zip(ORDER, ["#000000", "#000000", "#000000", "#000000", "#888888"]))
MARKERS = dict(zip(ORDER, ["o", "s", "o", "s", "D"]))
FACES = dict(zip(ORDER, ["#000000", "#000000", "white", "white", "#888888"]))
OFFSETS = [-0.28, -0.14, 0.0, 0.14, 0.28]
HEATS = ["day", "night", "compound"]
TICKS = ["Daytime-only", "Nighttime-only", "Compound"]
METRICS = [("bld_coverage_z", "Building density"), ("bld_mean_h_m_z", "Mean building height")]
ROWS = [("exposure", "log_exp_"), ("HWD", "log_hwd_")]
INFERENCE_NOTES = {"city_clustered": "city-clustered 95% CIs and significance",
                   "conventional": "conventional (unclustered) 95% CIs and significance"}
DAGGER = "\u2020"


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else DAGGER if p < 0.10 else ""


# ---------------- Load and check ----------------
raw = pd.read_csv(DML_RUN / "dml_all_results.csv")
d = raw[raw["w_variant"].eq("base") & raw["analysis"].eq("main_log1p")].copy()
assert d["status"].eq("ok").all(), "Some models were not estimated successfully"
assert d["transform"].eq("log1p").all() and set(d["scheme"]) == set(ORDER)
assert len(d) == len(ORDER) * 14 and not d.duplicated(["scheme", "treatment", "outcome"]).any()
assert d["n"].eq(4576).all() and d["n_clusters"].eq(35).all(), "The five definitions should use the same 4,576 grid-cell-years and 35 cities"
pop = d[d["outcome"].eq("log_pop")]
assert pop.groupby("treatment")["theta"].nunique().eq(1).all(), "Full-sample population estimates should not vary with the heatwave definition"
assert set(INFERENCES) <= set(INFERENCE_NOTES) and len(INFERENCES) >= 1
assert (d["cluster_df"] == d["n_clusters"] - 1).all()
cc, ci = t.ppf(0.975, d["cluster_df"]), t.ppf(0.975, d["n_used"] - 2)
d["CI_lower_city_clustered"], d["CI_upper_city_clustered"] = d["theta"] - cc * d["SE_main"], d["theta"] + cc * d["SE_main"]
d["p_city_clustered"] = 2 * t.sf(np.abs(d["theta"] / d["SE_main"]), d["cluster_df"])
d["CI_lower_conventional"], d["CI_upper_conventional"] = d["theta"] - ci * d["SE_iid"], d["theta"] + ci * d["SE_iid"]
d["p_conventional"] = 2 * t.sf(np.abs(d["theta"] / d["SE_iid"]), d["n_used"] - 2)
for mine, saved in [("CI_lower_city_clustered", "CI_lo_main"), ("CI_upper_city_clustered", "CI_hi_main"),
                    ("CI_lower_conventional", "CI_lo_iid"), ("CI_upper_conventional", "CI_hi_iid")]:
    np.testing.assert_allclose(d[mine], d[saved], atol=1e-9, rtol=0)
np.testing.assert_allclose(d["p_city_clustered"], d["pval_main"], atol=1e-9, rtol=1e-6)
np.testing.assert_allclose(d["p_conventional"], d["pval_iid"], atol=1e-9, rtol=1e-6)

# ---------------- Plotting ----------------
font_manager.findfont(font_manager.FontProperties(family="Arial"), fallback_to_default=False)
plt.rcParams.update({
    "font.family": "Arial", "font.size": 10, "axes.titlesize": 10,
    "axes.labelsize": 10, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})
OUT.mkdir(parents=True, exist_ok=True)
for inference in INFERENCES:
    lo_col, hi_col, p_col = f"CI_lower_{inference}", f"CI_upper_{inference}", f"p_{inference}"
    fig, axes = plt.subplots(len(ROWS), 2, figsize=FIGSIZE)
    plotted = []
    for row, (family, prefix) in enumerate(ROWS):
        outcomes = [prefix + h for h in HEATS]
        limits = XLIMS[family]
        pad = 0.012 * (limits[1] - limits[0])
        for col, (treatment, metric) in enumerate(METRICS):
            ax = axes[row, col]
            panel = PANEL_LETTERS[2 * row + col]
            for scheme, offset in zip(ORDER, OFFSETS):
                s = d[d["scheme"].eq(scheme) & d["treatment"].eq(treatment)].set_index("outcome").loc[outcomes]
                x, lo, hi, pv = (s[c].to_numpy() for c in ["theta", lo_col, hi_col, p_col])
                y = np.arange(len(HEATS)) + offset
                assert lo.min() > limits[0] and hi.max() < limits[1], f"X axis does not cover all intervals: {family} {treatment} {scheme}"
                ax.errorbar(x, y, xerr=np.vstack([x - lo, hi - x]),
                            fmt=MARKERS[scheme], color=COLORS[scheme],
                            markerfacecolor=FACES[scheme], markeredgewidth=1,
                            markersize=4.5, elinewidth=1.05, capsize=2.2,
                            linestyle="none", zorder=3)
                marks = [stars(v) for v in pv]
                for xx, yy, mk in zip(hi, y, marks):
                    if mk:
                        ax.text(xx + pad, yy + (0.0 if mk == DAGGER else 0.045), mk, ha="left", va="center",
                                fontsize=STAR_FONTSIZE, color=COLORS[scheme], clip_on=False, zorder=4)
                e = s.reset_index()[["scheme", "treatment", "treatment_label", "outcome", "theta",
                                     lo_col, hi_col, p_col, "n", "n_cells", "n_clusters"]].copy()
                e = e.rename(columns={lo_col: "CI_lower", hi_col: "CI_upper", p_col: "p_value"})
                e.insert(0, "panel", panel)
                e["definition"], e["family"], e["heatwave_type"] = LABELS[scheme], family, HEATS
                e["inference"], e["significance"] = inference, marks
                e["p_city_clustered"], e["p_conventional"] = s["p_city_clustered"].to_numpy(), s["p_conventional"].to_numpy()
                e["Y_position"] = y
                plotted.append(e)
            ax.axvline(0, color="0.65", linestyle="--", linewidth=0.8, zorder=1)
            ax.set_yticks(np.arange(len(HEATS)))
            ax.set_yticklabels(TICKS)
            ax.set_ylim(2.55, -0.55)
            ax.set_xlim(limits)
            ax.grid(axis="x", color="0.93", linewidth=0.6)
            ax.set_axisbelow(True)
            ax.set_title(f"{panel}  {metric}: {family}", loc="left", pad=9)
            ax.set_xlabel("DML coefficient (per 1 SD)")
    handles = [Line2D([0], [0], marker=MARKERS[s], color=COLORS[s], markerfacecolor=FACES[s],
                      linestyle="none", markersize=5, label=LABELS[s]) for s in ORDER]
    h = fig.get_figheight()
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.54, 1 - 0.02 / h),
               ncol=3, frameon=False, prop={"family": "Arial", "size": 9})
    note = ["Full sample, log(1 + x) outcomes; " + INFERENCE_NOTES[inference],
            f"*** P < 0.001, ** P < 0.01, * P < 0.05, {DAGGER} P < 0.10"]
    fig.text(0.54, 1 - 0.66 / h, ";  ".join(note), ha="center", va="center", fontsize=8.5, color="0.25")
    fig.subplots_adjust(left=0.14, right=0.955, bottom=0.60 / h, top=1 - 1.20 / h, hspace=0.64, wspace=0.58)

    plot_data = pd.concat(plotted, ignore_index=True)
    assert len(plot_data) == 60 and not plot_data.duplicated(["scheme", "treatment", "outcome"]).any()
    plot_data.to_csv(OUT / f"{FIG_NAME}_{inference}_plot_data.csv", index=False, encoding="utf-8-sig")
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(OUT / f"{FIG_NAME}_{inference}.{ext}", dpi=300, facecolor="white")
    plt.show()

print("Output:", OUT)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from scipy.stats import t

# ======================== User configuration ========================
ROOT = Path("..")
DML_RUN = Path("../outputs/dml")
OUT = ROOT / "outputs" / "figures" / "figS5_S6"
SCHEME = "local_p90"
FIG1_NAME = "Figure_DML_sensitivity_inference_and_adjustment"
FIG2_NAME = "Figure_DML_climate_groups"
PANEL_LETTERS = "ABCDEF"
STAR_FONTSIZE = 8.5
CLIMATE_COLORS = {"Tropical": "#2A9D8F", "Arid": "#E76F51", "Temperate": "#264653"}
FIGSIZE_1 = (9.5, 9.4)
XLIMS_1 = {"exposure": (-0.60, 0.98), "HWD": (-0.14, 0.12), "population": (-0.22, 1.10)}
XLIMS_2 = {"exposure": (-1.05, 1.80), "HWD": (-0.20, 0.17), "population": (-0.40, 1.48)}
# ==========================================================

HEATS = ["day", "night", "compound"]
TICKS = ["Daytime-only", "Nighttime-only", "Compound"]
METRICS = [("bld_coverage_z", "Building density"), ("bld_mean_h_m_z", "Mean building height")]
OUTCOME_ROWS = {"exposure": [f"log_exp_{h}" for h in HEATS], "HWD": [f"log_hwd_{h}" for h in HEATS], "population": ["log_pop"]}
DAGGER = "\u2020"


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else DAGGER if p < 0.10 else ""


def add_inference(d):
    d = d.copy()
    assert (d["cluster_df"] == d["n_clusters"] - 1).all()
    cc, ci = t.ppf(0.975, d["cluster_df"]), t.ppf(0.975, d["n_used"] - 2)
    d["lo_cluster"], d["hi_cluster"] = d["theta"] - cc * d["SE_main"], d["theta"] + cc * d["SE_main"]
    d["p_cluster"] = 2 * t.sf(np.abs(d["theta"] / d["SE_main"]), d["cluster_df"])
    d["lo_iid"], d["hi_iid"] = d["theta"] - ci * d["SE_iid"], d["theta"] + ci * d["SE_iid"]
    d["p_iid"] = 2 * t.sf(np.abs(d["theta"] / d["SE_iid"]), d["n_used"] - 2)
    for mine, saved in [("lo_cluster", "CI_lo_main"), ("hi_cluster", "CI_hi_main"), ("lo_iid", "CI_lo_iid"), ("hi_iid", "CI_hi_iid")]:
        np.testing.assert_allclose(d[mine], d[saved], atol=1e-9, rtol=0)
    np.testing.assert_allclose(d["p_cluster"], d["pval_main"], atol=1e-9, rtol=1e-6)
    np.testing.assert_allclose(d["p_iid"], d["pval_iid"], atol=1e-9, rtol=1e-6)
    return d


def forest(ax, sub, outcomes, ticks, series, limits, title, both=False):
    n_s = len(series)
    offsets = np.linspace(-0.28, 0.28, n_s) if len(outcomes) > 1 else np.linspace(-0.24, 0.24, n_s)
    pad = 0.012 * (limits[1] - limits[0])
    rows = []
    for (key, frame, style, inference), offset in zip(series, offsets):
        s = frame.set_index("outcome").loc[outcomes]
        x = s["theta"].to_numpy()
        lo, hi, pv = (s[f"{c}_{inference}"].to_numpy() for c in ["lo", "hi", "p"])
        y = np.arange(len(outcomes)) + offset
        assert lo.min() > limits[0] and hi.max() < limits[1], f"X axis does not cover all intervals: {title} {key}"
        ax.errorbar(x, y, xerr=np.vstack([x - lo, hi - x]), fmt=style["marker"], color=style["color"],
                    markerfacecolor=style["face"], markeredgewidth=1, markersize=4.5, elinewidth=1.05,
                    capsize=2.2, linestyle="none", zorder=3)
        marks = [stars(v) for v in pv]
        if both:
            for yy, mk, pi in zip(y, marks, s["p_iid"].to_numpy()):
                if not (mk or stars(pi)):
                    continue
                ax.text(1.03, yy, f"{mk or 'n.s.'} ({stars(pi) or 'n.s.'})", transform=ax.get_yaxis_transform(),
                        ha="left", va="center", fontsize=7.6, color=style["color"], clip_on=False, zorder=4)
        else:
            for xx, yy, mk in zip(hi, y, marks):
                if mk:
                    ax.text(xx + pad, yy + (0.0 if mk == DAGGER else 0.045), mk, ha="left", va="center",
                            fontsize=STAR_FONTSIZE, color=style["color"], clip_on=False, zorder=4)
        e = s.reset_index()[["outcome", "treatment", "treatment_label", "theta", "n", "n_cells", "n_clusters"]].copy()
        e["series"], e["CI_method"] = key, "City-clustered, t(G-1)" if inference == "cluster" else "Conventional, t(n-2)"
        e["CI_lower"], e["CI_upper"], e["p_value"], e["significance"] = lo, hi, pv, marks
        e["p_city_clustered"], e["p_conventional"] = s["p_cluster"].to_numpy(), s["p_iid"].to_numpy()
        e["Y_position"] = y
        rows.append(e)
    ax.axvline(0, color="0.65", linestyle="--", linewidth=0.8, zorder=1)
    ax.set_yticks(np.arange(len(outcomes)))
    ax.set_yticklabels(ticks)
    ax.set_ylim((0.5, -0.5) if len(outcomes) == 1 else (2.55, -0.55))
    ax.set_xlim(limits)
    ax.grid(axis="x", color="0.93", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_title(title, loc="left", pad=9)
    ax.set_xlabel("DML coefficient (per 1 SD)")
    if both:
        ax.text(1.03, 1.02, "Clustered" + chr(10) + "(conventional)", transform=ax.transAxes, ha="left", va="bottom",
                fontsize=7.6, color="0.25", linespacing=1.15, clip_on=False)
    return rows


# ---------------- Load and check ----------------
raw = pd.read_csv(DML_RUN / "dml_all_results.csv")
raw = raw[raw["scheme"].eq(SCHEME)]
main = add_inference(raw[raw["analysis"].eq("main_log1p")])
assert main["status"].eq("ok").all() and main["transform"].eq("log1p").all()
VARIANTS = {"base": "base", "no_mutual": "base_no_mutual_morphology", "context": "base_plus_context"}
for v in VARIANTS.values():
    g = main[main["w_variant"].eq(v)]
    assert len(g) == 14 and not g.duplicated(["treatment", "outcome"]).any(), f"Unexpected number of estimates in the main analysis: {v}"
assert main.loc[main["w_variant"].eq("base"), "n"].eq(4576).all() and main["n_clusters"].eq(35).all()
clim = add_inference(raw[raw["analysis"].eq("climate_log1p") & raw["w_variant"].eq("base")])
assert clim["status"].eq("ok").all() and len(clim) == 42
zone_names = {}
for z in sorted(clim["sample"].unique()):
    zone_names[z] = (DML_RUN / "blocks" / f"{SCHEME}__climate_log1p__base__{z}" / "climate_zone.txt").read_text(encoding="utf-8").strip()
assert set(zone_names.values()) == {"Arid (B)", "Tropical (A)", "Temperate/Highland (C)"}
clim["zone"] = clim["sample"].map(zone_names)
assert clim.groupby("zone")["n_clusters"].first().to_dict() == {"Arid (B)": 7, "Temperate/Highland (C)": 7, "Tropical (A)": 21}
assert clim.groupby("zone")["n"].first().sum() == 4576

mpl.rcParams.update(mpl.rcParamsDefault)
font_manager.findfont(font_manager.FontProperties(family="Arial"), fallback_to_default=False)
plt.rcParams.update({
    "font.family": "Arial", "font.size": 10, "axes.titlesize": 10, "axes.labelsize": 10,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})
OUT.mkdir(parents=True, exist_ok=True)
STAR_NOTE = f"*** P < 0.001, ** P < 0.01, * P < 0.05, {DAGGER} P < 0.10"

# ---------------- Figure 1: inference basis and control-variable specifications ----------------
SERIES_1 = [
    ("main_cluster", "base", "Main model", dict(marker="o", color="#000000", face="#000000"), "cluster"),
    ("no_mutual", "no_mutual", "No mutual adjustment", dict(marker="D", color="#555555", face="#555555"), "cluster"),
    ("context", "context", "Additional urban controls", dict(marker="^", color="#888888", face="#888888"), "cluster"),
]
fig, axes = plt.subplots(3, 2, figsize=FIGSIZE_1, gridspec_kw={"height_ratios": [1, 1, 0.55]})
plotted = []
for row, name in enumerate(["exposure", "HWD", "population"]):
    outcomes = OUTCOME_ROWS[name]
    ticks = ["Population"] if name == "population" else TICKS
    for col, (treatment, metric) in enumerate(METRICS):
        panel = PANEL_LETTERS[2 * row + col]
        series = [(key, main[main["w_variant"].eq(VARIANTS[var]) & main["treatment"].eq(treatment)], style, inf)
                  for key, var, _, style, inf in SERIES_1]
        rows = forest(axes[row, col], None, outcomes, ticks, series, XLIMS_1[name], f"{panel}  {metric}: {name}")
        plotted += [r.assign(panel=panel) for r in rows]
handles = [Line2D([0], [0], marker=st["marker"], color=st["color"], markerfacecolor=st["face"], linestyle="none",
                  markersize=5, label=lab) for _, _, lab, st, _ in SERIES_1]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.54, 0.998), ncol=3, frameon=False,
           prop={"family": "Arial", "size": 9})
fig.text(0.54, 0.945, "City-clustered 95% CIs and significance;  " + STAR_NOTE, ha="center", va="center",
         fontsize=8.5, color="0.25")
fig.subplots_adjust(left=0.14, right=0.955, bottom=0.065, top=0.872, hspace=0.64, wspace=0.58)
data1 = pd.concat(plotted, ignore_index=True)
assert len(data1) == len(SERIES_1) * 14 and not data1.duplicated(["series", "treatment", "outcome"]).any()
data1.to_csv(OUT / "sensitivity_plot_data.csv", index=False, encoding="utf-8-sig")
for ext in ["pdf", "svg", "png"]:
    fig.savefig(OUT / f"{FIG1_NAME}.{ext}", dpi=300, facecolor="white")
plt.show()

# ---------------- Figure 2: climate groups ----------------
ZONES = [("Arid (B)", "Arid", dict(marker="o", color=CLIMATE_COLORS["Arid"], face=CLIMATE_COLORS["Arid"])),
         ("Tropical (A)", "Tropical", dict(marker="s", color=CLIMATE_COLORS["Tropical"], face=CLIMATE_COLORS["Tropical"])),
         ("Temperate/Highland (C)", "Temperate", dict(marker="D", color=CLIMATE_COLORS["Temperate"], face=CLIMATE_COLORS["Temperate"]))]
rows2 = ["exposure", "HWD", "population"]
ratios = [1 if r != "population" else 0.5 for r in rows2]
fig, axes = plt.subplots(len(rows2), 2, figsize=(10.6, 2.55 * sum(ratios) + 2.0),
                         gridspec_kw={"height_ratios": ratios})
plotted = []
for row, name in enumerate(rows2):
    outcomes = OUTCOME_ROWS[name]
    ticks = ["Population"] if name == "population" else TICKS
    for col, (treatment, metric) in enumerate(METRICS):
        panel = PANEL_LETTERS[2 * row + col]
        series = [(short, clim[clim["zone"].eq(zone) & clim["treatment"].eq(treatment)], style, "cluster")
                  for zone, short, style in ZONES]
        rows = forest(axes[row, col], None, outcomes, ticks, series, XLIMS_2[name], f"{panel}  {metric}: {name}",
                      both=True)
        plotted += [r.assign(panel=panel) for r in rows]
ncity = clim.groupby("zone")["n_clusters"].first()
handles = [Line2D([0], [0], marker=st["marker"], color=st["color"], markerfacecolor=st["face"], linestyle="none",
                  markersize=5, label=f"{short} ({int(ncity[zone])} cities)") for zone, short, st in ZONES]
top = 1 - 0.95 / fig.get_figheight()
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.54, 0.998), ncol=3, frameon=False,
           prop={"family": "Arial", "size": 9})
note2 = "City-clustered 95% CIs;  significance: city-clustered (conventional);  " + STAR_NOTE
fig.text(0.54, 1 - 0.42 / fig.get_figheight(), note2, ha="center", va="center", fontsize=8.5, color="0.25")
fig.subplots_adjust(left=0.125, right=0.905, bottom=0.62 / fig.get_figheight(), top=top, hspace=0.70, wspace=0.74)
data2 = pd.concat(plotted, ignore_index=True)
assert len(data2) == 3 * 2 * sum(len(OUTCOME_ROWS[r]) for r in rows2)
data2.to_csv(OUT / "climate_groups_plot_data.csv", index=False, encoding="utf-8-sig")
for ext in ["pdf", "svg", "png"]:
    fig.savefig(OUT / f"{FIG2_NAME}.{ext}", dpi=300, facecolor="white")
plt.show()

print("Output:", OUT)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
import shap

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 10

# ======================== User configuration ========================
SHAP_TABLES = Path("../outputs/shap/tables")
OUT = Path("../outputs/figures/figS7_S8")
RANDOM_STATE = 42
# ==========================================================

KEY = ["city", "cell_id", "year"]
OUTCOMES = ["log_exp_day", "log_exp_night", "log_exp_compound"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


# ---------------- Load and check ----------------
features = pd.read_csv(SHAP_TABLES / "ml_sample_features.csv", float_precision="round_trip")
FEATS = [c for c in features.columns if c not in KEY + ["koppen_broad"]]
X_df = features[FEATS].astype(float)
koppen_arr = features.koppen_broad.values
shap_values = {}
for out in OUTCOMES:
    s = pd.read_csv(SHAP_TABLES / f"shap_values_xgboost_{out}.csv", float_precision="round_trip")
    require(s[KEY].equals(features[KEY]) and list(s.columns[len(KEY):]) == FEATS,
            f"SHAP values do not match the sample table: {out}")
    shap_values[out] = s[FEATS].to_numpy(float)

COL_DAY, COL_NIGHT, COL_COMPOUND = "#E8A33D", "#4E8AC2", "#A43A33"
MECH_COLORS = [COL_DAY, COL_NIGHT, COL_COMPOUND]
KOPPEN_ORDER = ["Tropical (A)", "Arid (B)", "Temperate/Highland (C)"]
KOPPEN_PAL = {"Tropical (A)": "#2A9D8F", "Arid (B)": "#E76F51", "Temperate/Highland (C)": "#264653"}
KOPPEN_SHOW = {"Tropical (A)": "Tropical", "Arid (B)": "Arid", "Temperate/Highland (C)": "Temperate"}
require(set(koppen_arr) == set(KOPPEN_ORDER), "Climate-zone names do not match the koppen_broad values")
FEATURE_LABELS = {"bld_coverage_z": "Building density", "bld_mean_h_m_z": "Mean building height",
                  "bld_std_h_m_z": "Building height SD", "lcz_1_z": "LCZ 1 (comp hi)",
                  "lcz_2_z": "LCZ 2 (comp mid)", "lcz_3_z": "LCZ 3 (comp lo)", "ndvi_mean_z": "NDVI",
                  "water_coverage_z": "Water coverage", "isa_mean_prob_z": "Impervious cover",
                  "cdd_22C_city": "CDD (22°C)", "rh_mean_pct": "Humidity", "tp_annual_mm": "Precip",
                  "lat": "Latitude", "dem_mean_m": "Elevation", "coast_dist_km": "Coast dist",
                  "koppen_B": "Köppen B", "koppen_C": "Köppen C",
                  "region_SA": "Region SA", "region_SEA": "Region SEA", "region_LA": "Region LA",
                  "year_num": "Year"}
OUT.mkdir(parents=True, exist_ok=True)

# ---------------- Figure S7: SHAP summary ----------------
short = {"log_exp_day": "Daytime-only", "log_exp_night": "Nighttime-only", "log_exp_compound": "Compound"}
X_plot = X_df.rename(columns=FEATURE_LABELS)
fig, axes_all = plt.subplots(2, 3, figsize=(10, 11.69))
for col_i, out in enumerate(OUTCOMES):
    sv = shap_values[out]
    ax_bee = axes_all[0, col_i]
    plt.sca(ax_bee)
    np.random.seed(RANDOM_STATE)  # the beeswarm shuffles the drawing order of the points
    shap.summary_plot(sv, X_plot, plot_type="dot", max_display=12, show=False, color_bar=(col_i == 2),
                      plot_size=None)
    ax_bee.set_title(short[out], fontsize=12, fontweight="bold", color=MECH_COLORS[col_i], pad=8)
    ax_bee.set_xlabel("SHAP value", fontsize=9)
    ax_bee.tick_params(labelsize=9)
    ax_bar = axes_all[1, col_i]
    plt.sca(ax_bar)
    shap.summary_plot(sv, X_plot, plot_type="bar", max_display=12, show=False, color_bar=False, plot_size=None)
    ax_bar.set_xlabel("Mean |SHAP value|", fontsize=9)
    ax_bar.tick_params(labelsize=9)
    if ax_bar.containers:
        ax_bar.bar_label(ax_bar.containers[0], fontsize=8, padding=2, fmt="%.2f")
axes_all[0, 0].set_ylabel("Beeswarm", fontsize=10)
axes_all[1, 0].set_ylabel("Feature importance", fontsize=10)
for col_i in [1, 2]:
    axes_all[0, col_i].set_ylabel("")
    axes_all[1, col_i].set_ylabel("")
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(OUT / "figS7_shap_summary.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "figS7_shap_summary.pdf", bbox_inches="tight")
plt.show()

# ---------------- Figure S8: SHAP dependence ----------------
SHOW_MORPH_DEP = ["bld_coverage_z", "bld_mean_h_m_z", "bld_std_h_m_z", "ndvi_mean_z", "water_coverage_z",
                  "isa_mean_prob_z"]
MORPH_LABELS_DEP = {"bld_coverage_z": "Building density", "bld_mean_h_m_z": "Mean building height",
                    "bld_std_h_m_z": "Building height SD", "ndvi_mean_z": "NDVI",
                    "water_coverage_z": "Water coverage", "isa_mean_prob_z": "Impervious surface cover"}
OUTCOMES_DEP = [("log_exp_day", "(a) Daytime-only exposure"), ("log_exp_night", "(b) Nighttime-only exposure"),
                ("log_exp_compound", "(c) Compound exposure")]

fig, axes = plt.subplots(6, 3, figsize=(8.27, 11.69))
n_drawn = {k: 0 for k in KOPPEN_ORDER}
for oi, (outcome_key, _) in enumerate(OUTCOMES_DEP):
    sv = shap_values[outcome_key]
    for vi, var in enumerate(SHOW_MORPH_DEP):
        ax = axes[oi * 2 + vi // 3, vi % 3]
        fi = X_df.columns.get_loc(var)
        xval, yval = X_df[var].values, sv[:, fi]
        for kopp in KOPPEN_ORDER:
            idx = np.where(koppen_arr == kopp)[0]
            if len(idx) < 10:
                continue
            n_drawn[kopp] += 1
            rng = np.random.default_rng(RANDOM_STATE)
            idx_plot = rng.choice(idx, size=min(1200, len(idx)), replace=False)
            ax.scatter(xval[idx_plot], yval[idx_plot], color=KOPPEN_PAL[kopp], alpha=0.10, s=5,
                       edgecolors="none", rasterized=True)
            o = np.argsort(xval[idx])
            xs, ys = xval[idx][o], yval[idx][o]
            win = max(25, len(xs) // 18)
            if len(xs) > win:
                rm = np.convolve(ys, np.ones(win) / win, mode="valid")
                ax.plot(xs[win // 2: win // 2 + len(rm)], rm, color=KOPPEN_PAL[kopp], lw=1.8)
        ax.axhline(0, color="black", lw=0.6, ls="--")
        ax.set_title(MORPH_LABELS_DEP[var], fontsize=8.2, pad=2)
        ax.set_xlabel("Standardised value (z-score)" if vi // 3 == 1 else "", fontsize=7.3)
        ax.set_ylabel("SHAP value" if vi % 3 == 0 else "", fontsize=7.3)
        ax.tick_params(axis="both", labelsize=6.8)
        ax.grid(alpha=0.20, linewidth=0.35)
        ax.set_axisbelow(True)
require(all(v == 18 for v in n_drawn.values()), f"Some climate zones were not drawn in the dependence plot: {n_drawn}")
for ypos, (_, title) in zip([0.83, 0.50, 0.17], OUTCOMES_DEP):
    fig.text(0.035, ypos, title, rotation=90, va="center", ha="center", fontsize=10, fontweight="bold")
fig.legend(handles=[Patch(facecolor=KOPPEN_PAL[k], edgecolor="none", label=KOPPEN_SHOW[k])
                    for k in KOPPEN_ORDER], loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.015),
           frameon=False, fontsize=8.5)
plt.tight_layout(rect=[0.08, 0.05, 0.99, 0.98])
plt.subplots_adjust(left=0.10, hspace=0.42, wspace=0.25)
fig.savefig(OUT / "figS8_shap_dependence.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "figS8_shap_dependence.pdf", bbox_inches="tight")
plt.show()
print("Output:", OUT)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D

# ======================== User configuration ========================
ROOT = Path("..")
MULTI_RUN = Path("../outputs/simulation")
OUT = ROOT / "outputs" / "figures" / "figS9"
FIG_NAME = "Figure_siting_sensitivity_18_settings"
FONT_FAMILY = "Arial"
FIGSIZE = (9.6, 7.6)
RULES = ["population", "density", "height", "random", "hwd_hist", "hwd_foresight"]
REFERENCE_RULES = {"hwd_foresight"}
HEAT_TYPES = ["any", "day", "night", "compound"]
# ==========================================================

SPLITS = [("main_2016-2019_to_2020-2023", "Selection 2016\u20132019, evaluation 2020\u20132023 (main)"),
          ("alt_2016-2018_to_2019-2023", "Selection 2016\u20132018, evaluation 2019\u20132023")]
DELTAS = [0.5, 1.0, 1.5]
BUDGETS = [0.1, 0.2, 0.3]
RULE_LABELS = {"population": "Population", "density": "Building density", "height": "Building height",
               "random": "Random", "hwd_hist": "Historical HWD", "hwd_foresight": "Concurrent HWD"}
RULE_COLORS = {"population": "#9C2E7F", "density": "#279A8D", "height": "#7A9A2E",
               "random": "#9A9A9A", "hwd_hist": "#2B7BB9", "hwd_foresight": "#E46F35"}
HEAT_LABELS = {"any": "Total exposure", "day": "Daytime-only", "night": "Nighttime-only", "compound": "Compound"}
TEXT = "#202124"


def summarize_settings(eff):
    keys = ["split", "heatwave_type", "cooling_mode", "delta_t_C", "budget_share", "strategy"]
    records = []
    for values, g in eff.groupby(keys, sort=True):
        total, oracle_total = float(g.avoided_exposure.sum()), float(g.oracle_avoided.sum())
        records.append(dict(zip(keys, values)) | dict(
            n_cities_all=int(g.city.nunique()),
            ratio_of_total_benefits=total / oracle_total if oracle_total > 0 else np.nan,
            total_avoided_million_person_days_per_year=float(g.avoided_person_days_per_year.sum() / 1e6),
            actual_cell_share_min=float(g.actual_cell_share.min()),
            actual_cell_share_max=float(g.actual_cell_share.max())))
    return pd.DataFrame(records)


# ---------------- Load and check ----------------
eff_file = MULTI_RUN / "multiyear_siting_efficiency.csv"
assert eff_file.is_file(), f"Simulation results not found: {eff_file}"
a = summarize_settings(pd.read_csv(eff_file))
a = a[a["cooling_mode"].eq("tmax_tmin") & a["strategy"].isin(RULES + ["oracle"])].copy()
keys = ["split", "heatwave_type", "delta_t_C", "budget_share"]
assert set(a["split"]) == {s for s, _ in SPLITS} and set(a["heatwave_type"]) == set(HEAT_TYPES)
assert a.groupby(keys).ngroups == len(SPLITS) * len(HEAT_TYPES) * len(DELTAS) * len(BUDGETS) == 72
assert a.groupby(keys)["strategy"].nunique().eq(len(RULES) + 1).all() and not a.duplicated(keys + ["strategy"]).any()
assert np.allclose(a.loc[a["strategy"].eq("oracle"), "ratio_of_total_benefits"], 1.0), "oracle should be 100%"
assert a["n_cities_all"].eq(35).all(), "All 35 cities should be included"
a["share_pct"] = 100 * a["ratio_of_total_benefits"]
data = a[a["strategy"].isin(RULES)].copy()
assert len(data) == 72 * len(RULES) == 432 and np.isfinite(data["share_pct"]).all()
assert data["share_pct"].between(0, 100 + 1e-9).all()

# ---------------- Plotting ----------------
mpl.rcParams.update(mpl.rcParamsDefault)
font_manager.findfont(font_manager.FontProperties(family=FONT_FAMILY), fallback_to_default=False)
plt.rcParams.update({
    "font.family": FONT_FAMILY, "font.size": 8.5, "axes.titlesize": 8.5, "axes.labelsize": 8.5,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.7, "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})
ncol = len(SPLITS) * len(DELTAS)
fig, axes = plt.subplots(len(HEAT_TYPES), ncol, figsize=FIGSIZE, sharex=True, sharey=True)
x = np.array([100 * b for b in BUDGETS])
for i, heat in enumerate(HEAT_TYPES):
    for j, ((split, _), delta) in enumerate([(s, dt) for s in SPLITS for dt in DELTAS]):
        ax = axes[i, j]
        g = data[data["split"].eq(split) & data["heatwave_type"].eq(heat) & np.isclose(data["delta_t_C"], delta)]
        if split == SPLITS[0][0] and np.isclose(delta, 1.0):
            ax.axvspan(15, 25, color="#EFEFEF", linewidth=0, zorder=0)
        ax.axhline(100, color="#555555", linestyle=(0, (1, 1.5)), linewidth=0.8, zorder=1)
        for rule in RULES:
            r = g[g["strategy"].eq(rule)].set_index("budget_share")
            yv = np.array([float(r.loc[[b for b in r.index if np.isclose(b, bb)][0], "share_pct"]) for bb in BUDGETS])
            ref = rule in REFERENCE_RULES
            ax.plot(x, yv, color=RULE_COLORS[rule], linewidth=1.15 if ref else 1.4,
                    linestyle=(0, (4, 2)) if ref else "-", marker="o", markersize=3.6,
                    markerfacecolor="white" if ref else RULE_COLORS[rule],
                    markeredgecolor=RULE_COLORS[rule], markeredgewidth=0.9, zorder=3 if ref else 4)
        ax.set_xlim(5, 35)
        ax.set_ylim(0, 108)
        ax.set_xticks(x)
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.grid(axis="y", color="#ECECEC", linewidth=0.55, zorder=0)
        ax.set_axisbelow(True)
        if i == 0:
            ax.set_title(f"{delta:.1f} \u00b0C cooling", fontsize=8.5, color=TEXT, pad=5)
        if j == 0:
            ax.set_ylabel(HEAT_LABELS[heat] + "\n(% of oracle benefit)", fontsize=8.5)
            ax.text(-0.62, 1.06, "ABCD"[i], transform=ax.transAxes, fontsize=10.5, fontweight="bold",
                    color=TEXT, ha="left", va="bottom")
        if i == len(HEAT_TYPES) - 1:
            ax.set_xlabel("Budget (% of cells)", fontsize=8.5)

fig.subplots_adjust(left=0.105, right=0.985, bottom=0.075, top=0.845, hspace=0.20, wspace=0.14)
for k, (_, label) in enumerate(SPLITS):
    left = axes[0, k * len(DELTAS)].get_position()
    right = axes[0, (k + 1) * len(DELTAS) - 1].get_position()
    fig.text((left.x0 + right.x1) / 2, left.y1 + 0.052, label, ha="center", va="bottom", fontsize=9, color=TEXT)
    fig.add_artist(Line2D([left.x0, right.x1], [left.y1 + 0.046, left.y1 + 0.046], color="#777777", linewidth=0.7))
handles = [Line2D([0], [0], color=RULE_COLORS[r], linewidth=1.15 if r in REFERENCE_RULES else 1.4,
                  linestyle=(0, (4, 2)) if r in REFERENCE_RULES else "-", marker="o", markersize=4,
                  markerfacecolor="white" if r in REFERENCE_RULES else RULE_COLORS[r],
                  markeredgecolor=RULE_COLORS[r], label=RULE_LABELS[r]) for r in RULES]
handles.append(Line2D([0], [0], color="#555555", linestyle=(0, (1, 1.5)), linewidth=0.8, label="Oracle"))
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.545, 0.998), ncol=len(handles), frameon=False,
           fontsize=8.5, handlelength=2.2, columnspacing=1.3, handletextpad=0.5)

# ---------------- Output ----------------
OUT.mkdir(parents=True, exist_ok=True)
export = data[keys + ["strategy", "share_pct", "total_avoided_million_person_days_per_year",
                      "n_cities_all", "actual_cell_share_min", "actual_cell_share_max"]].copy()
export.insert(4, "rule", export.pop("strategy").map(RULE_LABELS))
export = export.sort_values(keys + ["rule"]).reset_index(drop=True)
export.to_csv(OUT / "siting_settings_plot_data.csv", index=False, encoding="utf-8-sig")
for ext in ["pdf", "svg", "png"]:
    fig.savefig(OUT / f"{FIG_NAME}.{ext}", dpi=300, facecolor="white")
plt.show()

print("Output:", OUT)